#### antes de nada verificamos si detecta la gpu

In [1]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs detectadas:", gpus)

if gpus:
    print("Todo correcto: TensorFlow ve la RTX 3060")
else:
    print("Algo va mal: TF no detecta la GPU")

2026-05-17 15:25:20.826168: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-17 15:25:20.871047: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-17 15:25:20.884045: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-17 15:25:20.967296: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow: 2.17.0
GPUs detectadas: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Todo correcto: TensorFlow ve la RTX 3060


I0000 00:00:1779031524.953398     138 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1779031525.006805     138 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1779031525.008050     138 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355


## Script para analizar la base de datos, pretendemos hallar cuantos locutores,  PINS y defs

In [13]:
from pathlib import Path

RAIZ = Path("reconocedorLocutor_TF/PIN_16kHz")

locutores = sorted([d for d in RAIZ.iterdir() if d.is_dir()])
print(f"Locutores encontrados: {len(locutores)}\n")

total, total_def = 0, 0
por_locutor = {}

for loc in locutores:
    wavs = list(loc.glob("*.wav"))
    defs = [w for w in wavs if "(def)" in w.name]
    buenos = [w for w in wavs if "(def)" not in w.name]
    por_locutor[loc.name] = (len(buenos), len(defs))
    total += len(buenos)
    total_def += len(defs)

print(f"Audios válidos (sin def): {total}")
print(f"Audios marcados (def):    {total_def}\n")
print("Audios válidos por locutor:")
for nombre, (b, d) in por_locutor.items():
    print(f"  {nombre}: {b} válidos, {d} def")

Locutores encontrados: 24

Audios válidos (sin def): 2658
Audios marcados (def):    222

Audios válidos por locutor:
  093: 94 válidos, 26 def
  147: 120 válidos, 0 def
  291: 73 válidos, 47 def
  379: 118 válidos, 2 def
  527: 116 válidos, 4 def
  544: 120 válidos, 0 def
  626: 105 válidos, 15 def
  639: 117 válidos, 3 def
  774: 119 válidos, 1 def
  785: 111 válidos, 9 def
  920: 117 válidos, 3 def
  a77: 112 válidos, 8 def
  b48: 120 válidos, 0 def
  b73: 117 válidos, 3 def
  c09: 114 válidos, 6 def
  c70: 110 válidos, 10 def
  d42: 118 válidos, 2 def
  d65: 116 válidos, 4 def
  e11: 113 válidos, 7 def
  e41: 119 válidos, 1 def
  e85: 111 válidos, 9 def
  g38: 119 válidos, 1 def
  h76: 106 válidos, 14 def
  i22: 73 válidos, 47 def


In [14]:
import re
from collections import Counter

# NOTA: la frase/PIN se identifica con el contenido del paréntesis MÁS
# el dígito de variante que le sigue. Los "pin(3920)0", "pin(3920)1"... son
# PINs distintos de una misma familia, no el mismo PIN. La regex captura
# ambas partes. Además se corrige una errata detectada: "3290" -> "3920".
pins_reales = Counter()
for w in RAIZ.glob("*/*.wav"):
    if "(def)" in w.name:
        continue
    m = re.search(r"pin\(([^)]+)\)(\d*)", w.name)
    if m:
        clave = (m.group(1) + m.group(2)).replace("3290", "3920")
        pins_reales[clave] += 1

print(f"Clases de PIN reales: {len(pins_reales)}")
for pin, n in sorted(pins_reales.items()):
    print(f"  {pin}: {n} audios")

Clases de PIN reales: 40
  0185: 89 audios
  1763: 91 audios
  2311: 91 audios
  39200: 47 audios
  39201: 48 audios
  39202: 47 audios
  39203: 44 audios
  39204: 48 audios
  39205: 48 audios
  39206: 39 audios
  39207: 43 audios
  39208: 47 audios
  39209: 47 audios
  4057: 87 audios
  4103: 92 audios
  41730: 45 audios
  41731: 45 audios
  41732: 40 audios
  41733: 41 audios
  41734: 42 audios
  41735: 39 audios
  41736: 47 audios
  41737: 45 audios
  41738: 46 audios
  41739: 43 audios
  4447: 92 audios
  4565: 88 audios
  4860: 86 audios
  6068: 81 audios
  6154: 89 audios
  7382: 91 audios
  7621: 88 audios
  7919: 91 audios
  8214: 90 audios
  8913: 85 audios
  8936: 86 audios
  9169: 95 audios
  9218: 82 audios
  9355: 86 audios
  9501: 87 audios


Y para las duraciones, otra celda (mide una muestra)

In [15]:
import librosa, numpy as np

muestra = list(RAIZ.glob("*/*.wav"))[:200]
duraciones = []
for w in muestra:
    if "(def)" in w.name:
        continue
    d = librosa.get_duration(path=str(w))
    duraciones.append(d)

duraciones = np.array(duraciones)
print(f"Duración (sobre {len(duraciones)} audios):")
print(f"  min={duraciones.min():.2f}s  max={duraciones.max():.2f}s")
print(f"  media={duraciones.mean():.2f}s  percentil95={np.percentile(duraciones,95):.2f}s")

Duración (sobre 194 audios):
  min=1.20s  max=2.17s
  media=1.55s  percentil95=1.95s
